# **Greedy Modularity Optimization for Community Detection**

This notebook implements the **Greedy Modularity approach** for community detection in the Karate Club network.

## Overview
The greedy modularity algorithm:
- Starts with each node as an individual community
- Iteratively merges communities that increase modularity the most
- Produces fine-grained community structures
- Is computationally efficient for large networks
- Often discovers more communities than spectral methods

## Section 1: Import Required Libraries

Import all necessary libraries for network analysis, visualization, and data manipulation.

In [3]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

ModuleNotFoundError: No module named 'numpy'

## Section 2: Load and Prepare Network Data

Load the Karate Club graph from NetworkX and prepare it for modularity analysis.

In [ ]:
# Load the Karate Club graph
G = nx.karate_club_graph()

print(f"Network: {G}")
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print(f"Density: {nx.density(G):.4f}")

# Compute graph metrics
m = G.number_of_edges()  # Total number of edges
degrees = dict(G.degree())

print(f"\nDegree statistics:")
print(f"  Min degree: {min(degrees.values())}")
print(f"  Max degree: {max(degrees.values())}")
print(f"  Average degree: {sum(degrees.values()) / len(degrees):.2f}")

### Visualize the Original Network

In [ ]:
# Visualize the original network
fig, ax = plt.subplots(figsize=(12, 8))
pos = nx.spring_layout(G, seed=42, k=0.5, iterations=50)

nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=300, alpha=0.8, ax=ax)
nx.draw_networkx_edges(G, pos, alpha=0.3, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)

ax.set_title('Original Karate Club Network', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

## Section 3: Implement Greedy Modularity Algorithm

Implement the greedy modularity optimization algorithm from scratch.

In [ ]:
def calculate_modularity(graph, communities):
    """
    Calculate the modularity score of a partition.
    
    Q = (1/2m) * sum((A_ij - k_i*k_j/2m) * delta(c_i, c_j))
    
    Args:
        graph: NetworkX graph
        communities: List of lists, where each sublist contains nodes in a community
    
    Returns:
        Modularity score (float)
    """
    from networkx.algorithms import community as community_module
    try:
        return community_module.modularity(graph, communities)
    except:
        return 0.0


def greedy_modularity_optimization(graph):
    """
    Apply greedy modularity optimization algorithm.
    
    Algorithm:
    1. Start with each node as its own community
    2. For each pair of communities, calculate the modularity gain from merging
    3. Merge the pair that gives the maximum modularity gain
    4. Repeat until no positive gain is possible
    
    Args:
        graph: NetworkX graph
    
    Returns:
        communities: Final partition (list of lists)
        history: List of (iteration, modularity, num_communities)
    """
    nodes = list(graph.nodes())
    n = len(nodes)
    m = graph.number_of_edges()
    
    # Initialize: each node is its own community
    communities = {node: [node] for node in nodes}
    node_to_community = {node: node for node in nodes}
    
    history = []
    iteration = 0
    
    # Initial modularity
    current_communities = list(communities.values())
    current_modularity = calculate_modularity(graph, current_communities)
    history.append((iteration, current_modularity, len(communities)))
    best_modularity = current_modularity
    best_partition = [comm.copy() for comm in current_communities]
    
    print(f"Starting greedy modularity optimization...")
    print(f"Initial state: {len(communities)} communities, Q={current_modularity:.4f}")
    
    # Greedy merging loop
    while len(communities) > 1:
        community_ids = list(communities.keys())
        
        best_gain = 0
        best_pair = None
        
        # Find the pair of communities with the best modularity gain
        for i in range(len(community_ids)):
            for j in range(i + 1, len(community_ids)):
                comm_i = community_ids[i]
                comm_j = community_ids[j]
                
                # Merge communities i and j
                test_communities = communities.copy()
                test_communities[comm_i] = test_communities[comm_i] + test_communities[comm_j]
                del test_communities[comm_j]
                
                # Calculate modularity after merge
                test_partition = list(test_communities.values())
                test_modularity = calculate_modularity(graph, test_partition)
                gain = test_modularity - current_modularity
                
                if gain > best_gain:
                    best_gain = gain
                    best_pair = (comm_i, comm_j)
        
        # If no positive gain, stop
        if best_pair is None or best_gain <= 0:
            print(f"\nNo more beneficial merges. Stopping at iteration {iteration}.")
            print(f"Final state: {len(communities)} communities, Q={current_modularity:.4f}")
            break
        
        # Perform the best merge
        comm_i, comm_j = best_pair
        communities[comm_i] = communities[comm_i] + communities[comm_j]
        del communities[comm_j]
        
        # Update current state
        current_communities = list(communities.values())
        current_modularity = calculate_modularity(graph, current_communities)
        iteration += 1
        
        # Track best partition
        if current_modularity > best_modularity:
            best_modularity = current_modularity
            best_partition = [comm.copy() for comm in current_communities]
        
        history.append((iteration, current_modularity, len(communities)))
        
        if iteration % 5 == 0 or iteration <= 3:
            print(f"Iteration {iteration}: {len(communities)} communities, Q={current_modularity:.4f}, gain={best_gain:.6f}")
    
    return best_partition, history, best_modularity


print("Greedy modularity functions defined successfully.")

## Section 4: Detect Communities Using Greedy Algorithm

In [ ]:
# Apply greedy modularity optimization
print("="*60)
print("APPLYING GREEDY MODULARITY OPTIMIZATION")
print("="*60)

communities, history, best_modularity = greedy_modularity_optimization(G)

print(f"\n" + "="*60)
print("RESULTS SUMMARY")
print("="*60)
print(f"Number of communities detected: {len(communities)}")
print(f"Best modularity achieved: {best_modularity:.4f}")
print(f"\nCommunities:")
for idx, comm in enumerate(sorted(communities, key=len, reverse=True)):
    print(f"  Community {idx+1}: {sorted(comm)} (size: {len(comm)})")

## Section 5: Analyze Modularity Evolution

In [ ]:
# Extract history data
iterations = [h[0] for h in history]
modularity_scores = [h[1] for h in history]
num_communities = [h[2] for h in history]

# Create modularity evolution plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Modularity vs Iteration
ax1.plot(iterations, modularity_scores, 'o-', linewidth=2.5, markersize=8, color='#2E86AB')
ax1.fill_between(iterations, modularity_scores, alpha=0.2, color='#2E86AB')
best_iter = modularity_scores.index(max(modularity_scores))
ax1.plot(best_iter, max(modularity_scores), 'r*', markersize=20, label=f'Best Q={max(modularity_scores):.4f}')
ax1.set_xlabel('Iteration', fontsize=12, fontweight='bold')
ax1.set_ylabel('Modularity (Q)', fontsize=12, fontweight='bold')
ax1.set_title('Modularity Evolution During Greedy Merging', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.legend(fontsize=11)

# Plot 2: Number of Communities vs Iteration
ax2.plot(iterations, num_communities, 's-', linewidth=2.5, markersize=8, color='#A23B72')
ax2.fill_between(iterations, num_communities, alpha=0.2, color='#A23B72')
ax2.set_xlabel('Iteration', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of Communities', fontsize=12, fontweight='bold')
ax2.set_title('Community Count During Greedy Merging', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.set_xticks(iterations[::max(1, len(iterations)//10)])

plt.tight_layout()
plt.show()

# Print detailed history
print("\nDetailed Iteration History:")
print(f"{'Iteration':<12} {'Modularity':<15} {'# Communities':<15}")
print("-" * 42)
for iteration, mod, n_comm in history:
    print(f"{iteration:<12} {mod:<15.6f} {n_comm:<15}")

## Section 6: Visualize Community Structure

Create network visualizations showing the detected communities.

In [ ]:
# Create node-to-community mapping
node_to_community = {}
for comm_idx, community in enumerate(communities):
    for node in community:
        node_to_community[node] = comm_idx

# Visualize the detected communities
fig, ax = plt.subplots(figsize=(14, 10))

# Create color map for communities
num_colors = len(communities)
colors = plt.cm.Set3(np.linspace(0, 1, num_colors))
node_colors = [colors[node_to_community[node]] for node in G.nodes()]

# Draw the network
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=400, alpha=0.8, ax=ax)
nx.draw_networkx_edges(G, pos, alpha=0.3, width=1.5, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold', ax=ax)

# Add title and legend
ax.set_title(f'Greedy Modularity Communities (Q={best_modularity:.4f})', 
             fontsize=14, fontweight='bold')

# Create legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors[i], label=f'Community {i+1} ({len(communities[i])} nodes)')
                   for i in range(len(communities))]
ax.legend(handles=legend_elements, loc='upper left', fontsize=10)
ax.axis('off')
plt.tight_layout()
plt.show()

## Section 7: Calculate Community Statistics

Analyze the properties of detected communities.

In [ ]:
print("\n" + "="*60)
print("COMMUNITY STATISTICS")
print("="*60)

community_stats = []
for idx, community in enumerate(sorted(communities, key=len, reverse=True)):
    subgraph = G.subgraph(community)
    
    # Internal edges
    internal_edges = subgraph.number_of_edges()
    
    # External edges
    external_edges = 0
    for node in community:
        for neighbor in G.neighbors(node):
            if neighbor not in community:
                external_edges += 1
    
    # Density
    if len(community) > 1:
        max_edges = len(community) * (len(community) - 1) / 2
        density = internal_edges / max_edges if max_edges > 0 else 0
    else:
        density = 0
    
    # Average degree within community
    avg_degree = 2 * internal_edges / len(community) if len(community) > 0 else 0
    
    community_stats.append({
        'ID': idx + 1,
        'Size': len(community),
        'Internal Edges': internal_edges,
        'External Edges': external_edges,
        'Density': density,
        'Avg Degree': avg_degree,
        'Nodes': sorted(community)
    })

# Print statistics
print(f"\n{'ID':<5} {'Size':<8} {'Int Edges':<12} {'Ext Edges':<12} {'Density':<10} {'Avg Deg':<10}")
print("-" * 60)
for stat in community_stats:
    print(f"{stat['ID']:<5} {stat['Size']:<8} {stat['Internal Edges']:<12} {stat['External Edges']:<12} "
          f"{stat['Density']:<10.4f} {stat['Avg Degree']:<10.4f}")

# Overall statistics
print(f"\n{'='*60}")
print(f"Total communities: {len(communities)}")
print(f"Best modularity: {best_modularity:.4f}")
print(f"Average community size: {sum(len(c) for c in communities) / len(communities):.2f}")
print(f"Smallest community: {min(len(c) for c in communities)}")
print(f"Largest community: {max(len(c) for c in communities)}")

## Section 8: Compare Greedy vs Spectral Methods

Summary comparison with spectral bipartition approach.

In [ ]:
print("\n" + "="*60)
print("GREEDY MODULARITY OPTIMIZATION - FINAL SUMMARY")
print("="*60)

summary_data = f"""
ALGORITHM: Greedy Modularity Optimization

METHODOLOGY:
  • Starts with each node as an individual community
  • Iteratively merges pairs of communities
  • Selects merges that maximize modularity gain
  • Continues until no positive gain is possible

ADVANTAGES:
  ✓ Computationally efficient O(n log n) to O(n²)
  ✓ Discovers fine-grained community structures
  ✓ Often finds more communities than spectral methods
  ✓ Practical for large networks
  ✓ Greedy approach is intuitive and interpretable

LIMITATIONS:
  ✗ Can be trapped in local optima
  ✗ May over-partition networks
  ✗ Order-dependent in some implementations
  ✗ Less stable than exhaustive optimization

RESULTS FOR KARATE CLUB:
  • Communities detected: {len(communities)}
  • Best modularity (Q): {best_modularity:.4f}
  • Merging iterations: {len(history) - 1}
  • Final community sizes: {[len(c) for c in communities]}

INTERPRETATION:
  The greedy algorithm successfully partitioned the Karate Club network
  into {len(communities)} communities with modularity Q={best_modularity:.4f}.
  This represents a strong community structure where nodes within communities
  are more densely connected than expected by chance.

COMPARISON WITH SPECTRAL METHODS:
  • Greedy typically finds more communities
  • Greedy may have similar or higher modularity
  • Spectral methods are more deterministic
  • Both methods are complementary approaches
"""

print(summary_data)